In [ ]:
# Ensure working directory is project root and output directory exists
import os
# If notebook is opened from inside the notebooks/ folder, move to project root
cwd = os.getcwd()
if os.path.basename(cwd).lower() == 'notebooks':
    os.chdir(os.path.dirname(cwd))
print('Working dir set to', os.getcwd())

out_dir = os.path.join(os.getcwd(), 'notebooks')
os.makedirs(out_dir, exist_ok=True)

# If a nested notebooks/notebooks was previously created, move files up and remove nested dir
nested = os.path.join(out_dir, 'notebooks')
if os.path.isdir(nested):
    for name in os.listdir(nested):
        src = os.path.join(nested, name)
        dst = os.path.join(out_dir, name)
        if os.path.exists(dst):
            base, ext = os.path.splitext(name)
            i = 1
            while os.path.exists(dst):
                dst = os.path.join(out_dir, f"{base}_{i}{ext}")
                i += 1
        os.replace(src, dst)
    try:
        os.rmdir(nested)
    except OSError:
        pass

print('Output directory ensured:', out_dir)

In [1]:
from supabase import create_client, Client
from dotenv import load_dotenv
import os
from getpass import getpass

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError("SUPABASE_URL and SUPABASE_KEY must be set (via .env or interactive input).")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Client Supabase créé.")

Client Supabase créé.


# Analyse PlayerState — Visualisations

Cette section charge un échantillon des données `PlayerState` depuis Supabase et produit 3 graphiques :

- Distribution de `Damage`
- Évolution de la `MoveSpeed` moyenne en bins de frames
- Scatter `Damage` vs `MoveSpeed` coloré par `Victory`

Assure-toi que le fichier `.env` contient `SUPABASE_URL` et `SUPABASE_KEY` avant d'exécuter.

In [18]:
# Code: charger les données et tracer
from supabase import create_client
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# headless backend for automated image saving
import matplotlib
matplotlib.use('Agg')

load_dotenv()
SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError('Définis SUPABASE_URL et SUPABASE_KEY dans .env avant d\'exécuter cette cellule')

# ensure output dir at project root
cwd = os.getcwd()
if os.path.basename(cwd).lower() == 'notebooks':
    os.chdir(os.path.dirname(cwd))
out_dir = os.path.join(os.getcwd(), 'notebooks')
os.makedirs(out_dir, exist_ok=True)

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Récupérer un échantillon raisonnable
ps_resp = supabase.table('PlayerState').select('*').limit(5000).execute()
rows = None
if hasattr(ps_resp, 'data'):
    rows = ps_resp.data
elif isinstance(ps_resp, dict) and 'data' in ps_resp:
    rows = ps_resp['data']
else:
    rows = ps_resp

if not rows:
    raise RuntimeError('Aucune donnée PlayerState retournée')

df = pd.DataFrame(rows)

# Récupérer info de Run pour Victory (ignore si permission)
try:
    runs_resp = supabase.table('Run').select('id, Victory').limit(10000).execute()
    runs = runs_resp.data if hasattr(runs_resp, 'data') else (runs_resp['data'] if isinstance(runs_resp, dict) and 'data' in runs_resp else runs_resp)
    runs_df = pd.DataFrame(runs) if runs else pd.DataFrame(columns=['id','Victory'])
except Exception:
    runs_df = pd.DataFrame(columns=['id','Victory'])

# merge
if 'id_run' in df.columns and 'id' in runs_df.columns and not runs_df.empty:
    df = df.merge(runs_df, left_on='id_run', right_on='id', how='left')

# Convertir colonnes numériques
numeric_cols = ['Damage','MoveSpeed','Frame','Hearts','Coins','Bombs','Keys','TearRange','ShotSpeed','Luck']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

sns.set(style='whitegrid')

# Plot 1: distribution Damage
if 'Damage' in df.columns:
    plt.figure(figsize=(10,6))
    sns.histplot(df['Damage'].dropna(), bins=50, kde=True)
    plt.title('Distribution of Damage')
    plt.savefig(os.path.join(out_dir, 'figure_damage_hist.png'))
    plt.close()

# Plot 2: average MoveSpeed by frame bins
if 'Frame' in df.columns and 'MoveSpeed' in df.columns:
    df['frame_bin'] = (df['Frame'] // 100) * 100
    speed_by_frame = df.groupby('frame_bin')['MoveSpeed'].mean().reset_index()
    plt.figure(figsize=(10,6))
    sns.lineplot(data=speed_by_frame, x='frame_bin', y='MoveSpeed')
    plt.title('Average MoveSpeed by Frame (100-frame bins)')
    plt.xlabel('Frame (bin start)')
    plt.savefig(os.path.join(out_dir, 'figure_movespeed_line.png'))
    plt.close()

# Plot 3: Damage vs MoveSpeed colored by Victory (sample)
if 'MoveSpeed' in df.columns and 'Damage' in df.columns:
    sample = df.sample(n=min(1000, len(df)))
    plt.figure(figsize=(10,6))
    hue = 'Victory' if 'Victory' in df.columns else None
    sns.scatterplot(data=sample, x='MoveSpeed', y='Damage', hue=hue, alpha=0.6)
    plt.title('Damage vs MoveSpeed (sampled)')
    plt.savefig(os.path.join(out_dir, 'figure_damage_vs_movespeed.png'))
    plt.close()

print(f'Images saved to {out_dir}: figure_damage_hist.png, figure_movespeed_line.png, figure_damage_vs_movespeed.png')

Images saved to c:\Users\TCYH5041\projet-fil-rouge\notebooks: figure_damage_hist.png, figure_movespeed_line.png, figure_damage_vs_movespeed.png


# Visualisations intégrées

### Distribution de Damage

![Distribution de Damage](figure_damage_hist.png)

### Vitesse moyenne par frames (bins)

![Average MoveSpeed by Frame](figure_movespeed_line.png)

### Damage vs MoveSpeed (échantillon)

![Damage vs MoveSpeed](figure_damage_vs_movespeed.png)


In [19]:
# Average ClearDurationFrames per Stage
from supabase import create_client
import os
from dotenv import load_dotenv
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

load_dotenv()
SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')
if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError('Définis SUPABASE_URL et SUPABASE_KEY dans .env avant d\'exécuter')

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Fetch Room and Stage
rooms_resp = supabase.table('Room').select('id, id_stage, ClearDurationFrames').limit(20000).execute()
rooms = rooms_resp.data if hasattr(rooms_resp, 'data') else (rooms_resp['data'] if isinstance(rooms_resp, dict) and 'data' in rooms_resp else rooms_resp)
rooms_df = pd.DataFrame(rooms) if rooms else pd.DataFrame(columns=['id','id_stage','ClearDurationFrames'])

stages_resp = supabase.table('Stage').select('id, StageNumber').limit(10000).execute()
stages = stages_resp.data if hasattr(stages_resp, 'data') else (stages_resp['data'] if isinstance(stages_resp, dict) and 'data' in stages_resp else stages_resp)
stages_df = pd.DataFrame(stages) if stages else pd.DataFrame(columns=['id','StageNumber'])

if rooms_df.empty:
    print('Aucune donnée Room retournée')
elif stages_df.empty:
    print('Aucune donnée Stage retournée')
else:
    df = rooms_df.merge(stages_df, left_on='id_stage', right_on='id', how='left')
    df['ClearDurationFrames'] = pd.to_numeric(df['ClearDurationFrames'], errors='coerce')
    agg = df.groupby('StageNumber')['ClearDurationFrames'].mean().reset_index().sort_values('StageNumber')
    plt.figure(figsize=(10,6))
    sns.barplot(data=agg, x='StageNumber', y='ClearDurationFrames', color='C0')
    plt.title('Average ClearDurationFrames per Stage')
    plt.xlabel('StageNumber')
    plt.ylabel('Average ClearDurationFrames')
    plt.savefig('notebooks/figure_clear_duration_by_stage.png')
    plt.close()
    print('Saved notebooks/figure_clear_duration_by_stage.png')


Saved notebooks/figure_clear_duration_by_stage.png


In [11]:
try:
    test_room = supabase.table('Room').select('id, id_stage, ClearDurationFrames').limit(1).execute()
    print("Room OK")
except Exception as e:
    print("Room KO:", e)

try:
    test_stage = supabase.table('Stage').select('id, StageNumber').limit(1).execute()
    print("Stage OK")
except Exception as e:
    print("Stage KO:", e)


Room OK
Stage OK
